In [ ]:
import sys
print(sys.executable)

import torch, nltk, pickle
from torch import nn
from collections import Counter
from transformers import BatchEncoding, PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutput

import numpy as np
import sys, time, os

TRAIN_FILE = "./train.txt"
VAL_FILE = "./val.txt"
TOKENIZER_FILE = "tokenizer_file.pkl"
MODEL_OUTPUT_DIR = "./a1-rnn-model"

VOCABULARY_SIZE = 5000 #None # None means use all unique training tokens, plus the special tokens.
MODEL_MAX_LENGTH = 100

EMBEDDING_SIZE = 128
HIDDEN_SIZE = 128

LEARNING_RATE = 1e-3
NUM_TRAIN_EPOCHS = 5
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
USE_CPU = False

/Users/filipn/Documents/dev/WASP-DL-NLP/env/bin/python


# Assignment 1

In [24]:
###
### Part 1. Tokenization.
###

class A1Tokenizer:
    """A minimal implementation of a tokenizer similar to tokenizers in the HuggingFace library."""

    def __init__(self, str_to_int, int_to_str, tokenize_fun, model_max_length, pad_token,
                 unk_token, bos_token, eos_token):
        # store all values you need in order to implement __call__ below.
        self.str_to_int = str_to_int
        self.int_to_str = int_to_str
        self.tokenize_fun = tokenize_fun
        self.model_max_length = model_max_length # Needed for truncation.

        self.pad_token_id = str_to_int[pad_token] # Compulsory attribute.
        self.unk_token_id = str_to_int[unk_token]
        self.bos_token_id = str_to_int[bos_token]
        self.eos_token_id = str_to_int[eos_token]


    def __call__(self, texts, truncation=False, padding=False, return_tensors=None):
        """Tokenize the given texts and return a BatchEncoding containing the integer-encoded tokens.
           
           Args:
             texts:           The texts to tokenize.
             truncation:      Whether the texts should be truncated to model_max_length.
             padding:         Whether the tokenized texts should be padded on the right side.
             return_tensors:  If None, then return lists; if 'pt', then return PyTorch tensors.

           Returns:
             A BatchEncoding where the field `input_ids` stores the integer-encoded texts.
        """
        if return_tensors and return_tensors != 'pt':
            raise ValueError('Should be pt')
        
        # TODO: Your work here is to split the texts into words and map them to integer values.
        # 
        # - If `truncation` is set to True, the length of the encoded sequences should be 
        #   at most self.model_max_length.
        # - If `padding` is set to True, then all the integer-encoded sequences should be of the
        #   same length. That is: the shorter sequences should be "padded" by adding dummy padding
        #   tokens on the right side.
        # - If `return_tensors` is undefined, then the returned `input_ids` should be a list of lists.
        #   Otherwise, if `return_tensors` is 'pt', then `input_ids` should be a PyTorch 2D tensor.

        # Return a BatchEncoding where input_ids stores the result of the integer encoding.
        # Optionally, if you want to be 100% HuggingFace-compatible, you should also include an 
        # attention mask of the same shape as input_ids. In this mask, padding tokens correspond
        # to the the value 0 and real tokens to the value 1.
        encoded_texts = []
        for text in texts:
            word_tokens = self.tokenize_fun(text)
            token_ids = [self.bos_token_id]
            for word_token in word_tokens:
                token_id = self.str_to_int.get(word_token, self.unk_token_id)
                token_ids.append(token_id)
            token_ids.append(self.eos_token_id)
            if truncation and self.model_max_length is not None:
                token_ids = token_ids[:self.model_max_length]
                token_ids[-1] = self.eos_token_id
            encoded_texts.append(token_ids)

        if padding:
            longest_length = max(len(token_ids) for token_ids in encoded_texts)
            for token_ids in encoded_texts:
                number_of_padding_tokens = longest_length - len(token_ids)
                token_ids.extend([self.pad_token_id] * number_of_padding_tokens)

        attention_mask = [[0 if token_id == self.pad_token_id else 1 for token_id in token_ids] for token_ids in encoded_texts]

        if return_tensors == 'pt':
            encoded_texts = torch.tensor(encoded_texts)
            attention_mask = torch.tensor(attention_mask)

        return BatchEncoding({
            'input_ids': encoded_texts,
            'attention_mask': attention_mask
        })
        # return BatchEncoding({'input_ids': ...})

    def __len__(self):
        """Return the size of the vocabulary."""
        return len(self.str_to_int)
    
    def save(self, filename):
        """Save the tokenizer to the given file."""
        with open(filename, 'wb') as f:
            pickle.dump(self, f)

    @staticmethod
    def from_file(filename):
        """Load a tokenizer from the given file."""
        with open(filename, 'rb') as f:
            return pickle.load(f)


In [25]:
def lowercase_tokenizer(text):
    return [t.lower() for t in nltk.word_tokenize(text)]


def build_tokenizer(train_file, tokenize_fun=lowercase_tokenizer, max_voc_size=None, model_max_length=None,
                    pad_token='<PAD>', unk_token='<UNK>', bos_token='<BOS>', eos_token='<EOS>'):
    """ Build a tokenizer from the given file.

        Args:
             train_file:        The name of the file containing the training texts.
             tokenize_fun:      The function that maps a text to a list of string tokens.
             max_voc_size:      The maximally allowed size of the vocabulary.
             model_max_length:  Truncate texts longer than this length.
             pad_token:         The dummy string corresponding to padding.
             unk_token:         The dummy string corresponding to out-of-vocabulary tokens.
             bos_token:         The dummy string corresponding to the beginning of the text.
             eos_token:         The dummy string corresponding to the end the text.
    """
    # build the vocabulary, possibly truncating it to max_voc_size if that is specified.
    # Then return a tokenizer object (implemented below).
    special_tokens = [pad_token, unk_token, bos_token, eos_token]
    counter = Counter()
    with open(train_file, encoding='utf-8') as file:
        for line in file:
            text = line.strip()
            if text: counter.update(tokenize_fun(text))
    # print(counter)

    str_to_int = {}
    for token in special_tokens:
        str_to_int[token] = len(str_to_int)

    if max_voc_size is None:
        num_regular_tokens = None
    else:
        num_regular_tokens = max_voc_size - len(special_tokens)
        if num_regular_tokens < 0:
            raise ValueError("max_voc_size must be at least the number of special tokens")
    
    for token, _ in counter.most_common(num_regular_tokens):
          if token not in str_to_int:
              str_to_int[token] = len(str_to_int)

    # print(str_to_int)
    if max_voc_size: assert len(str_to_int) <= max_voc_size, "max_voc_size exeeded"

    int_to_str = {i: token for token, i in str_to_int.items()}

    return A1Tokenizer(
        str_to_int=str_to_int,
        int_to_str=int_to_str,
        tokenize_fun=tokenize_fun,
        model_max_length=model_max_length,
        pad_token=pad_token,
        unk_token=unk_token,
        bos_token=bos_token,
        eos_token=eos_token,
    )


print(lowercase_tokenizer("Let's test!!"))
tokenizer = build_tokenizer(train_file=TRAIN_FILE, max_voc_size=VOCABULARY_SIZE, model_max_length=MODEL_MAX_LENGTH)
print(len(tokenizer))


test_texts = ['This is a test.', 'Another test.']
result = tokenizer(test_texts, return_tensors='pt', padding=True, truncation=True)
print(result)
for encoded_text in result["input_ids"]: # type: ignore
    decoded_tokens = [tokenizer.int_to_str[token_id.item()] for token_id in encoded_text]
    print(decoded_tokens)

tokenizer.save(TOKENIZER_FILE)

['let', "'s", 'test', '!', '!']
5000
{'input_ids': tensor([[  2,  35,  14,  11, 975,   6,   3],
        [  2, 155, 975,   6,   3,   0,   0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0]])}
['<BOS>', 'this', 'is', 'a', 'test', '.', '<EOS>']
['<BOS>', 'another', 'test', '.', '<EOS>', '<PAD>', '<PAD>']


In [26]:
from datasets import load_dataset

dataset = load_dataset('text', data_files={'train': TRAIN_FILE, 'val': VAL_FILE})
dataset = dataset.filter(lambda x: x['text'].strip() != '')

# print(len(dataset["train"]))
# print(len(dataset["val"]))
# from torch.utils.data import Subset
# for sec in ['train', 'val']:
#     dataset[sec] = Subset(dataset[sec], range(1000)) # type: ignore

print(len(dataset["train"]))
print(len(dataset["val"]))

147059
17874


In [27]:
from torch.utils.data import DataLoader

dl = DataLoader(dataset['train'], batch_size=1, shuffle=True) # type: ignore

for batch in dl:
    print(batch)
    break

{'text': ["After leaving the coaching ranks immediately following his team's victory in Super Bowl XXIII, Walsh went to work as a broadcaster for NBC, teaming with Dick Enberg to form the lead broadcasting team, replacing Merlin Olsen."]}


In [28]:
###
### Part 3. Defining the model.
###

class A1RNNModelConfig(PretrainedConfig):
    """Configuration object that stores hyperparameters that define the RNN-based language model."""
    def __init__(self, vocab_size=1000, embedding_size=128, hidden_size=256, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.embedding_size = embedding_size

class A1RNNModel(PreTrainedModel):
    """The neural network model that implements a RNN-based language model."""
    config_class = A1RNNModelConfig
    
    def __init__(self, config):
        super().__init__(config)
        self.embedding = nn.Embedding(config.vocab_size, config.embedding_size)
        self.rnn = nn.LSTM(input_size=config.embedding_size, hidden_size=config.hidden_size, batch_first=True)
        self.unembedding = nn.Linear(config.hidden_size, config.vocab_size)
        # Note: -100 is the value HuggingFace conventionally uses to refer to tokens
        # where we do not want to compute the loss.
        self.loss_func = torch.nn.CrossEntropyLoss(ignore_index=-100)


    def forward(self, input_ids, labels=None):
        """The forward pass of the RNN-based language model.
        
           Args:
             - input_ids:  The input tensor (2D), consisting of a batch of integer-encoded texts.
             - labels:     The reference tensor (2D), consisting of a batch of integer-encoded texts.
           Returns:
             A CausalLMOutput containing
               - logits:   The output tensor (3D), consisting of logits for all token positions for all vocabulary items.
               - loss:     The loss computed on this batch.               
        """
        embedded = self.embedding(input_ids)
        rnn_out, _ = self.rnn(embedded)
        logits = self.unembedding(rnn_out)
        loss = None
        if labels is not None:
            shifted_logits = logits[:, :-1, :]
            shifted_labels = labels[:, 1:] # Ignore <BOS>
            loss = self.loss_func(shifted_logits.reshape(-1, shifted_logits.shape[-1]), shifted_labels.reshape(-1))

        return CausalLMOutput(logits=logits, loss=loss)


# Sanity check
config = A1RNNModelConfig(vocab_size=len(tokenizer), hidden_size=HIDDEN_SIZE, embedding_size=EMBEDDING_SIZE)
model = A1RNNModel(config)

N = 10
input_ids = torch.randint(low=0, high=len(tokenizer), size=(1, N))
output = model(input_ids)
print(input_ids.shape)
print(output.logits.shape)

torch.Size([1, 10])
torch.Size([1, 10, 5000])


In [29]:
from transformers import TrainingArguments

###
### Part 4. Training the language model.
###

## Hint: the following TrainingArguments hyperparameters may be relevant for your implementation:
#
# - optim:            What optimizer to use. You can assume that this is set to 'adamw_torch',
#                     meaning that we use the PyTorch AdamW optimizer.
# - eval_strategy:    You can assume that this is set to 'epoch', meaning that the model should
#                     be evaluated on the validation set after each epoch
# - use_cpu:          Force the trainer to use the CPU; otherwise, CUDA or MPS should be used.
#                     (In your code, you can just use the provided method select_device.)
# - learning_rate:    The optimizer's learning rate.
# - num_train_epochs: The number of epochs to use in the training loop.
# - per_device_train_batch_size: 
#                     The batch size to use while training.
# - per_device_eval_batch_size:
#                     The batch size to use while evaluating.
# - output_dir:       The directory where the trained model will be saved.



class A1Trainer:
    """A minimal implementation similar to a Trainer from the HuggingFace library."""

    def __init__(self, model, args, train_dataset, eval_dataset, tokenizer):
        """Set up the trainer.
           
           Args:
             model:          The model to train.
             args:           The training parameters stored in a TrainingArguments object.
             train_dataset:  The dataset containing the training documents.
             eval_dataset:   The dataset containing the validation documents.
             eval_dataset:   The dataset containing the validation documents.
             tokenizer:      The tokenizer.
        """
        self.model = model
        self.args = args
        self.train_dataset = train_dataset
        self.eval_dataset = eval_dataset
        self.tokenizer = tokenizer

        assert(args.optim == 'adamw_torch')
        assert(args.eval_strategy == 'epoch')

    def select_device(self):
        """Return the device to use for training, depending on the training arguments and the available backends."""
        if self.args.use_cpu:
            return torch.device('cpu')
        if not self.args.no_cuda and torch.cuda.is_available():
            return torch.device('cuda')
        if torch.mps.is_available():
            return torch.device('mps')
        return torch.device('cpu')
            
    def train(self):
        """Train the model."""
        args = self.args

        device = self.select_device()
        print('Device:', device)
        self.model.to(device)
        
        # The model already owns a loss function so we compute loss there instead, that is why I do not use this
        # loss_func = torch.nn.CrossEntropyLoss(ignore_index=self.tokenizer.pad_token_id)

        # Relevant arguments: at least args.learning_rate, but you can optionally also consider
        # other Adam-related hyperparameters here.
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=args.learning_rate)

        # Relevant arguments: args.per_device_train_batch_size, args.per_device_eval_batch_size
        train_loader = DataLoader(self.train_dataset, batch_size=args.per_device_train_batch_size, shuffle=True) # type: ignore
        val_loader = DataLoader(self.eval_dataset, batch_size=args.per_device_eval_batch_size, shuffle=False) # type: ignore
        
        # Your work here is to implement the training loop.
        #       
        # for each training epoch (use args.num_train_epochs here):
        #   for each batch B in the training set:
        #
        #       PREPROCESSING AND FORWARD PASS:
        #       input_ids = apply your tokenizer to B
        #       labels = input_ids with padding replaced by -100
	    #       put input_ids and labels onto the GPU (or whatever device you use)
        #       apply the model to input_ids and labels
        #       get the loss from the model output
        #
        #       BACKWARD PASS AND MODEL UPDATE:
        #       optimizer.zero_grad()
        #       loss.backward()
        #       optimizer.step()
        for epoch in range(int(args.num_train_epochs)):
            self.model.train()
            total_loss = 0.0
            for batch in train_loader:
                encoded = self.tokenizer(batch["text"], truncation=True, padding=True, return_tensors="pt")
                input_ids = encoded["input_ids"]
                labels = input_ids.clone()
                labels[labels == self.tokenizer.pad_token_id] = -100
                input_ids = input_ids.to(device)
                labels = labels.to(device)

                output = self.model(input_ids=input_ids, labels=labels)
                loss = output.loss

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            
            avg_train_loss = total_loss / len(train_loader)
            total_val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    encoded = self.tokenizer(batch["text"], truncation=True, padding=True, return_tensors="pt")
                    input_ids = encoded["input_ids"]
                    labels = input_ids.clone()
                    labels[labels == self.tokenizer.pad_token_id] = -100
                    input_ids = input_ids.to(device)
                    labels = labels.to(device)
                    output = self.model(input_ids=input_ids, labels=labels)
                    total_val_loss += output.loss.item()
            avg_val_loss = total_val_loss / len(val_loader)
            print(f"Epoch {epoch + 1}: "
                  f"train loss = {avg_train_loss:.4f}, "
                  f"val loss = {avg_val_loss:.4f}")

        print(f"Saving to {args.output_dir}.")
        self.model.save_pretrained(args.output_dir)


def train():
    config = A1RNNModelConfig(vocab_size=len(tokenizer), hidden_size=HIDDEN_SIZE, embedding_size=EMBEDDING_SIZE)
    args = TrainingArguments(output_dir=MODEL_OUTPUT_DIR,
                            optim="adamw_torch",
                            eval_strategy="epoch",
                            learning_rate=LEARNING_RATE,
                            num_train_epochs=NUM_TRAIN_EPOCHS,
                            per_device_train_batch_size=TRAIN_BATCH_SIZE,
                            per_device_eval_batch_size=EVAL_BATCH_SIZE,
                            use_cpu=USE_CPU)
    model = A1RNNModel(config)
    A1Trainer(model=model,
            args=args,
            train_dataset=dataset["train"],
            eval_dataset=dataset["val"],
            tokenizer=tokenizer).train()
    
# train()

In [ ]:
text = "She lives in San"

model = A1RNNModel.from_pretrained(MODEL_OUTPUT_DIR)
device = next(model.parameters()).device
encoded = tokenizer([text], truncation=True, padding=False, return_tensors="pt")
input_ids = encoded["input_ids"].to(device)

model.eval()
with torch.no_grad():
    output = model(input_ids=input_ids)

# Skip EOS
next_token_logits = output.logits[0, -2, :]

top = torch.topk(next_token_logits, 10)

for token_id, score in zip(top.indices.cpu(), top.values.cpu()):
    print(tokenizer.int_to_str[token_id.item()], score.item())

In [ ]:
model = A1RNNModel.from_pretrained(MODEL_OUTPUT_DIR)
model.eval() # Not needed I think but good practice as I understand it

val_loader = DataLoader(dataset["val"], batch_size=EVAL_BATCH_SIZE, shuffle=False) # type: ignore

total_loss = 0.0
total_tokens = 0
with torch.no_grad():
    for batch in val_loader:
        encoded = tokenizer(batch["text"], truncation=True, padding=True, return_tensors="pt")
        input_ids = encoded["input_ids"]
        labels = input_ids.clone()
        labels[labels == tokenizer.pad_token_id] = -100

        output = model(input_ids=input_ids, labels=labels)

        shifted_labels = labels[:, 1:] # Ignore <BOS>
        num_tokens = (shifted_labels != -100).sum().item()
        total_loss += output.loss.item() * num_tokens
        total_tokens += num_tokens

mean_loss = total_loss / total_tokens
perplexity = torch.exp(torch.tensor(mean_loss)).item()

print(f"Validation loss: {mean_loss:.4f}")
print(f"Validation perplexity: {perplexity:.2f}")

In [ ]:
import torch.nn.functional as F

def nearest_neighbors(model, tokenizer, word, n_neighbors=5):
    if word not in tokenizer.str_to_int:
        print(f"{word!r} is not in the vocabulary.")
        return []
    embeddings = model.embedding.weight
    word_id = tokenizer.str_to_int[word]
    word_embedding = embeddings[word_id]

    cosine_scores = F.cosine_similarity(word_embedding.unsqueeze(0), embeddings, dim=1)
    cosine_scores[word_id] = -float("inf")
    scores, token_ids = cosine_scores.topk(n_neighbors)
    return [(tokenizer.int_to_str[token_id.item()], score.item())
            for token_id, score in zip(token_ids, scores)]


model = A1RNNModel.from_pretrained(MODEL_OUTPUT_DIR)
model.eval()
for word in ["human", "france", "city", "music"]:
    print(f"\nNearest neighbors of {word!r}:")

    neighbors = nearest_neighbors(model, tokenizer, word)
    for neighbor, similarity in neighbors:
        print(f"  {neighbor:15s} {similarity:.3f}")

# Assignment 2

In [30]:
import torch
from torch import nn
import torch.nn.functional as F
from transformers import PreTrainedModel, PretrainedConfig
from transformers.modeling_outputs import CausalLMOutput

class A2ModelConfig(PretrainedConfig):
    """Configuration object that stores hyperparameters that define the Transformer language model."""
    def __init__(self, vocab_size=None, hidden_size=None, intermediate_size=None, num_attention_heads=None, 
                 num_hidden_layers=None,
                 rope_theta=None, hidden_act='silu', max_position_embeddings=None, rms_norm_eps=None, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.rms_norm_eps = rms_norm_eps
        self.num_attention_heads = num_attention_heads
        self.rope_theta = rope_theta
        self.hidden_act = hidden_act
        self.intermediate_size = intermediate_size
        self.num_hidden_layers = num_hidden_layers



class A2MLP(nn.Module):
    """The MLP layer of the Transformer. Uses the SwiGLU architecture."""
    def __init__(self, config):
        super().__init__()
        assert(config.hidden_act == 'silu')
        # initalize components here
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.activation = nn.SiLU()

    def forward(self, hidden_states):
        return self.down_proj(self.activation(self.gate_proj(hidden_states)) * self.up_proj(hidden_states))
    

config = A2ModelConfig(hidden_size=128, intermediate_size=256, hidden_act="silu")
mlp = A2MLP(config)
test_input = torch.randn(2, 10, 128) # batch_size, sequence_length, hidden_size
test_output = mlp(test_input)

print("Input shape: ", test_input.shape)
print("Output shape:", test_output.shape)
assert test_output.shape == test_input.shape

Input shape:  torch.Size([2, 10, 128])
Output shape: torch.Size([2, 10, 128])


In [31]:
#### RoPE implementation (copied and simplified from HuggingFace). ####
def apply_rotary_pos_emb(q, k, rope_rotations, unsqueeze_dim=1):
    """Applies precomputed RoPE rotations to the query and key representations."""
    assert(q.shape == k.shape)
    assert(len(q.shape) == 4)
    cos, sin = rope_rotations
    assert(q.shape[2] == cos.shape[1])
    assert(q.shape[3] == cos.shape[2])    
    q_type, k_type = q.dtype, k.dtype
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed.to(q_type), k_embed.to(k_type)

def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

class A2RotaryEmbedding(nn.Module):
    """RoPE position representation for use in Transformer attention."""

    def __init__(self, config, device=None):
        super().__init__()
        rope_theta = config.rope_theta
        head_dim = config.hidden_size // config.num_attention_heads
        partial_rotary_factor = 1.0
        dim = int(head_dim * partial_rotary_factor)
        self.inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2, dtype=torch.int64).to(device=device, dtype=torch.float) / dim))

    @torch.no_grad()
    def forward(self, x):
        position_ids = torch.arange(0, x.shape[1], device=x.device).unsqueeze(0)
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1).to(x.device)
        position_ids_expanded = position_ids[:, None, :].float()

        device_type = x.device.type if isinstance(x.device.type, str) and x.device.type != "mps" else "cpu"
        with torch.autocast(device_type=device_type, enabled=False):  # Force float32
            freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
            emb = torch.cat((freqs, freqs), dim=-1)
            cos = emb.cos()
            sin = emb.sin()
            return cos, sin



class A2Attention(nn.Module):
    """The multi-head attention layer of the Transformer. Uses standard scaled dot-product attention with causal masking."""
    
    def __init__(self, config):
        super().__init__()
        assert config.hidden_size % config.num_attention_heads == 0
        self.num_heads = config.num_attention_heads
        self.hidden_size = config.hidden_size
        self.head_dim = self.hidden_size // self.num_heads
        # set up W_q, W_k, W_v, W_o here
        self.W_q = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_k = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_v = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        self.W_o = nn.Linear(config.hidden_size, config.hidden_size, bias=False)
        # set up normalizers here
        self.q_norm = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps, elementwise_affine=True)
        self.k_norm = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps, elementwise_affine=True)


    def forward(self, hidden_states, rope_rotations):
        b, m, d = hidden_states.shape
        query_representation = self.q_norm(self.W_q(hidden_states))
        key_representation = self.k_norm(self.W_k(hidden_states))
        value_representation = self.W_v(hidden_states)
        q = query_representation.view(b, m, self.num_heads, self.head_dim).transpose(1, 2)
        k = key_representation.view(b, m, self.num_heads, self.head_dim).transpose(1, 2)
        v = value_representation.view(b, m, self.num_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rotary_pos_emb(q, k, rope_rotations)
        attention_output = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attention_output = attention_output.transpose(1, 2).reshape(b, m, d)
        return self.W_o(attention_output)




# Sanity check
config = A2ModelConfig(hidden_size=128, num_attention_heads=4, rms_norm_eps=1e-5, rope_theta=10000)
attention = A2Attention(config)
test_input_ids = torch.zeros(2, 10, dtype=torch.long)
rope_rotations = A2RotaryEmbedding(config)(test_input_ids)
test_input = torch.randn(2, 10, 128)
output = attention(test_input, rope_rotations)
print(output)

tensor([[[ 0.1777,  0.0107, -0.3427,  ..., -0.0801,  0.0164, -0.0643],
         [ 0.0662, -0.0908, -0.2469,  ..., -0.1964, -0.0974,  0.1057],
         [ 0.1291, -0.0365, -0.1702,  ..., -0.2153, -0.0608,  0.2736],
         ...,
         [ 0.0039,  0.0131,  0.0438,  ...,  0.0749,  0.1782,  0.2916],
         [ 0.2325,  0.0974,  0.0092,  ...,  0.1867,  0.0649,  0.2123],
         [ 0.0524,  0.0559, -0.1123,  ...,  0.1234,  0.2563,  0.1225]],

        [[-0.2831, -0.0549, -0.1194,  ...,  0.0780,  0.0087,  0.2336],
         [-0.2101, -0.0825, -0.2208,  ..., -0.0985, -0.0363, -0.0063],
         [-0.1174, -0.1882, -0.1597,  ..., -0.1993, -0.1919, -0.1763],
         ...,
         [-0.0636,  0.2789, -0.0548,  ..., -0.0621,  0.2096, -0.2476],
         [-0.0606,  0.2296,  0.1028,  ...,  0.1718, -0.2195, -0.2923],
         [ 0.0458,  0.1663,  0.0442,  ..., -0.0180,  0.1403, -0.0651]]],
       grad_fn=<UnsafeViewBackward0>)


In [32]:
class A2DecoderLayer(nn.Module):
    """A complete Transformer decoder layer."""
    def __init__(self, config):
        super().__init__()
        # set up attention, MLP, and normalizers here.
        self.attention = A2Attention(config)
        self.mlp = A2MLP(config)
        self.attention_norm = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps, elementwise_affine=True)
        self.mlp_norm = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps, elementwise_affine=True)


    def forward(self, hidden_states, rope_rotations):
        attention_output = self.attention(hidden_states, rope_rotations)
        hidden_states = hidden_states + self.attention_norm(attention_output)
        mlp_output = self.mlp(hidden_states)
        hidden_states = hidden_states + self.mlp_norm(mlp_output)
        return hidden_states


class A2Transformer(PreTrainedModel):
    """A language model based on the Transformer architecture."""
    
    config_class = A2ModelConfig

    def __init__(self, config):
        super().__init__(config)
        self.rotary_emb = A2RotaryEmbedding(config)
        # Set up the other components here.
        self.embedding = nn.Embedding(config.vocab_size, config.hidden_size)
        self.final_norm = nn.RMSNorm(config.hidden_size, eps=config.rms_norm_eps, elementwise_affine=True)
        self.unembedding = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.loss_func = nn.CrossEntropyLoss(ignore_index=-100)
        # put all transformer decoder layers in a ModuleList.
        self.layers = nn.ModuleList([A2DecoderLayer(config)
                                     for _ in range(config.num_hidden_layers)])
        # This line should be called after you have set up all components.
        self.post_init()


    def forward(self, input_ids, labels=None):
        rope_rotations = self.rotary_emb(input_ids) # pass this to all the transformer decoder layers
        
        # Call embedding, transformer decoder layers, last normalizer, and unembedding.
        hidden_states = self.embedding(input_ids)
        for layer in self.layers:
            hidden_states = layer(hidden_states, rope_rotations)
        hidden_states = self.final_norm(hidden_states)
        logits = self.unembedding(hidden_states)

        # Compute the loss as in Assignment 1 if labels is not None.
        loss = None
        if labels is not None:
            shifted_logits = logits[:, :-1, :]
            shifted_labels = labels[:, 1:]
            loss = self.loss_func(shifted_logits.reshape(-1, shifted_logits.shape[-1]),
                                  shifted_labels.reshape(-1))
        return CausalLMOutput(logits=logits, loss=loss)


config = A2ModelConfig(vocab_size=1000,
                       hidden_size=128,
                       intermediate_size=256,
                       num_attention_heads=4,
                       num_hidden_layers=2,
                       rope_theta=10000,
                       rms_norm_eps=1e-5,
                       hidden_act="silu")

model = A2Transformer(config)
input_ids = torch.randint(low=0, high=config.vocab_size, size=(2, 10)) # type: ignore
output = model(input_ids)

print("Input shape: ", input_ids.shape)
print("Logits shape:", output.logits.shape)
assert output.logits.shape == (2, 10, 1000)

Input shape:  torch.Size([2, 10])
Logits shape: torch.Size([2, 10, 1000])


In [33]:
from transformers import TrainingArguments

TRANSFORMER_OUTPUT_DIR = "./a2-transformer-model"

config = A2ModelConfig(vocab_size=len(tokenizer),
                       hidden_size=128,
                       intermediate_size=256,
                       num_attention_heads=4,
                       num_hidden_layers=2,
                       rope_theta=10000,
                       rms_norm_eps=1e-5,
                       hidden_act="silu")

model = A2Transformer(config)

args = TrainingArguments(output_dir=TRANSFORMER_OUTPUT_DIR,
                         optim="adamw_torch",
                         eval_strategy="epoch",
                         learning_rate=1e-3,
                         num_train_epochs=2,
                         per_device_train_batch_size=32,
                         per_device_eval_batch_size=32,
                         use_cpu=False)

trainer = A1Trainer(model=model,
                    args=args,
                    train_dataset=dataset["train"],
                    eval_dataset=dataset["val"],
                    tokenizer=tokenizer)

trainer.train()


Device: mps
Epoch 1: train loss = 4.2063, val loss = 3.8988
Epoch 2: train loss = 3.8318, val loss = 3.7898
Saving to ./a2-transformer-model.


In [34]:
TRANSFORMER_OUTPUT_DIR = "./a2-transformer-model"

model = A2Transformer.from_pretrained(TRANSFORMER_OUTPUT_DIR)
model.eval()

total_parameters = sum(p.numel() for p in model.parameters())
print(f"Total parameters:     {total_parameters:,}")




text = "he lives in san"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device) # type: ignore
model.eval()

encoded = tokenizer([text],
                    truncation=True,
                    padding=False,
                    return_tensors="pt")

input_ids = encoded["input_ids"].to(device) # type: ignore
with torch.no_grad():
    output = model(input_ids=input_ids)

next_token_logits = output.logits[0, -2, :] # Tokenizer adds <EOS>

print("\nTop predictions:")
top = torch.topk(next_token_logits, k=10)
for token_id, score in zip(top.indices.cpu(), top.values.cpu()):
    word = tokenizer.int_to_str[token_id.item()]
    print(f"{word:15} {score.item():.3f}")

Total parameters:     1,608,832

Top predictions:
<UNK>           11.022
francisco       9.788
diego           8.288
salvador        7.767
in              5.115
of              5.015
johnson         4.996
,               4.961
de              4.940
and             4.828


In [42]:
from torch.distributions import Categorical

def generate_text(model, tokenizer, prompt, max_length=30, temperature=1.0, topk=20):
    if temperature <= 0:
        raise ValueError("temperature must be greater than 0")
    
    device = next(model.parameters()).device
    model.eval()

    encoded = tokenizer([prompt], truncation=True, padding=False, return_tensors="pt")
    input_ids = encoded["input_ids"].to(device)

    # Remove the <EOS> added by the tokenizer because generation should continue after the prompt.
    input_ids = input_ids[:, :-1]
    with torch.no_grad():
        for _ in range(max_length):
            output = model(input_ids=input_ids)
            next_token_logits = output.logits[0, -1, :]
            next_token_logits[tokenizer.pad_token_id] = -float("inf")
            next_token_logits[tokenizer.bos_token_id] = -float("inf")

            next_token_logits = next_token_logits / temperature # Temperature scaling
            
            # Keep only the top-k candidates
            k = min(topk, next_token_logits.shape[0])
            top_logits, top_token_ids = torch.topk(next_token_logits, k=k)
            # Sample among the top-k candidates
            distribution = Categorical(logits=top_logits)
            sampled_position = distribution.sample()
            next_token_id = top_token_ids[sampled_position]

            if next_token_id.item() == tokenizer.eos_token_id:
                break

            input_ids = torch.cat([input_ids, next_token_id.reshape(1, 1)], dim=1)

    generated_tokens = [tokenizer.int_to_str[token_id.item()]
                        for token_id in input_ids[0]
                        if token_id.item() not in {tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id}]

    return " ".join(generated_tokens)

## Run It

model = A2Transformer.from_pretrained("./a2-transformer-model")
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device) # type: ignore



prompt = "Is Paris the capital of France? Answer yes or no. The answer is "

generated = generate_text(model=model,
                          tokenizer=tokenizer,
                          prompt=prompt,
                          max_length=30,
                          temperature=1.0,
                          topk=20)

print(generated)


for temperature in [0.5, 0.7, 1.0]:
    print(f"\nTemperature: {temperature}")
    print(generate_text(model, tokenizer, prompt=prompt, max_length=30, temperature=temperature, topk=20))


is paris the capital of france ? answer <UNK> or no . the answer is to <UNK> :

Temperature: 0.5
is paris the capital of france ? answer <UNK> or no . the answer is to the <UNK> of the <UNK> <UNK> . it is the <UNK> of the <UNK> <UNK> and <UNK> <UNK> . it is sometimes <UNK> to <UNK> <UNK> , <UNK> by

Temperature: 0.7
is paris the capital of france ? answer <UNK> or no . the answer is <UNK> to the authority of the <UNK> , and its <UNK> must be <UNK> .

Temperature: 1.0
is paris the capital of france ? answer <UNK> or no . the answer is the <UNK> , <UNK> the land of the land . <UNK> will not follow a <UNK> <UNK> <UNK> :


In [45]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "allenai/OLMo-2-0425-1B"
olmo_tokenizer = AutoTokenizer.from_pretrained(model_name)
olmo_model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
olmo_model = olmo_model.to(device) # type: ignore



def generate_with_olmo(prompt, max_new_tokens=50, temperature=1.0, topk=20):
    encoded = olmo_tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        generated_ids = olmo_model.generate(**encoded, 
                                            max_new_tokens=max_new_tokens,
                                            do_sample=True,
                                            temperature=temperature,
                                            top_k=topk,
                                            pad_token_id=olmo_tokenizer.eos_token_id)

    return olmo_tokenizer.decode(generated_ids[0], skip_special_tokens=True)





prompts = ["In natural language processing, a Transformer",
           "Is Paris the capital of France? Answer yes or no. The answer is",
           "Write a Python program that reverses a list."]
for prompt in prompts:
    print("\nPROMPT:")
    print(prompt)

    print(generate_with_olmo(prompt, max_new_tokens=50, temperature=0.7, topk=20))
    print("\n" + "=" * 80)


Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.14s/it]



PROMPT:
In natural language processing, a Transformer
In natural language processing, a Transformer is a special type of neural network that is used to read and understand sentences and translate them into other languages. The Transformer is designed to be a model that learns from the data and improves with experience, which is a key feature that sets it apart from


PROMPT:
Is Paris the capital of France? Answer yes or no. The answer is
Is Paris the capital of France? Answer yes or no. The answer is no. Paris is not the capital of France. It is the city of light. It is not the capital of Europe. The capital of France is Paris. It is not the city of light. It is the city of love. It is also


PROMPT:
Write a Python program that reverses a list.
Write a Python program that reverses a list. The list is reversed in such a way that it starts with the first item of the list at the beginning of the list and moves to the last item at the end of the list. For example, if the input is [1, 2,

